In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") 

# Pydantic

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000120FE8F55D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000120FE8F6090>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="title of the movie")
    year:int=Field(description="the year the movie is released")
    director:str=Field(description="Name of the director")
    rating:float=Field(description="rating of the movie")
    

In [12]:
model_with_structure=model.with_structured_output(Movie,include_raw=True)
model_with_structure

{
  raw: RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000120FE8F55D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000120FE8F6090>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'title of the movie', 'type': 'string'}, 'year': {'description': 'the year the movie is released', 'type': 'integer'}, 'director': {'description': 'Name of the director', 'type': 'string'}, 'rating': {'description': 'rating of the movie', 'type': 'number'}}, 'required': ['title', 'year', 'direc

In [13]:
model_with_structure.invoke("Provide the details about the movie kingkong shull island")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie "King Kong: Skull Island". Likely need to use function Movie to get details (director, rating, title, year). Let\'s call function.', 'tool_calls': [{'id': 'fc_adb72f0b-58d4-4708-828e-3f61b3dabd10', 'function': {'arguments': '{"director":"Jordan Vogt-Roberts","rating":7,"title":"Kong: Skull Island","year":2017}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 157, 'total_tokens': 255, 'completion_time': 0.215711897, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.05194762, 'prompt_tokens_details': None, 'queue_time': 0.252865369, 'total_time': 0.267659517}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cd681-f95a-73e1-9557-f27b0d171c1

# Nested Structure

In [16]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str


class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres :list[str]
    budget: float| None = Field(None, description="Budet in usd")

In [19]:
model_with_nested_structure=model.with_structured_output(MovieDetails,include_raw=True)
model_with_nested_structure

{
  raw: RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000120FE8F55D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000120FE8F6090>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': '', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'cast': {'items': {'properties': {'name': {'type': 'string'}, 'role': {'type': 'string'}}, 'required': ['name', 'role'], 'type': 'object'}, 'type': 'array'}, 'genres': {'items': {'type': 'string'}, 'type': 'array'}, 'budget': {'anyOf': [{'type': 'number'}

In [20]:
model_with_nested_structure.invoke("Provide the details about the movie kingkong shull island")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need details about the movie "King Kong: Skull Island". Likely want title, year, genres, cast, budget. We can use function MovieDetails. Provide info. Let\'s call function.', 'tool_calls': [{'id': 'fc_cde38112-c1c2-43f7-8754-4f781df7a9a6', 'function': {'arguments': '{"budget":185000000,"cast":[{"name":"Tom Hiddleston","role":"James Conrad"},{"name":"Samuel L. Jackson","role":"Mason Weaver"},{"name":"John C. Reilly","role":"Jack Chapman"},{"name":"Brie Larson","role":"Mona Banks"},{"name":"John Goodman","role":"Bill Randa"},{"name":"Peter Stormare","role":"Colonel Preston Packard"},{"name":"Corey Stoll","role":"Hank Marlow"},{"name":"Jason Mitchell","role":"Hank Marlow"},{"name":"John Boyega","role":"Cole"},{"name":"Sung Kang","role":"Mason Weaver\'s teammate"},{"name":"Max Martini","role":"Mason Weaver\'s teammate"},{"name":"Toby Kebbell","role":"Mason Weaver\'s teammate"}],"genres":["Action","Adventure","Fantasy